In [3]:
print(123)

123


In [4]:
from ingest import load_faq_data
documents = load_faq_data()

In [5]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

85

In [6]:
documents = documents_llm

In [7]:
documents[10]

{'id': '20c5a1347e',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Where can I track the LLM Zoomcamp syllabus, deadlines, homework, and progress?',
 'answer': 'Use the [LLM Zoomcamp course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nIt contains the current cohort structure, homework, deadlines, and progress tracking. The process is the same as in other DataTalks.Club Zoomcamps.'}

In [8]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [10]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
import json

user_prompt = json.dumps(doc)

SyntaxError: unmatched ')' (1274931766.py, line 4)

In [12]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [14]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [18]:
result = response.output_parsed

print(result)

questions=['I just found this course — is it too late to join now?', 'Can I still start the course if I’m discovering it late?', 'If I join after the course has started, can I still get a certificate?', 'What do I need to do to be eligible for the course certificate?', 'Are project submissions only accepted while the course is still open?']


In [19]:
print(result.questions)

['I just found this course — is it too late to join now?', 'Can I still start the course if I’m discovering it late?', 'If I join after the course has started, can I still get a certificate?', 'What do I need to do to be eligible for the course certificate?', 'Are project submissions only accepted while the course is still open?']


In [20]:
from evaluation_utils import llm_structured

In [21]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Can I still join the course if I found it late?', 'If I join after the course started, am I still eligible for a certificate?', 'What do I need to do to get the certificate if I’m joining now?', 'Is it okay to start the course after it has already begun?', 'Do I have to submit the project before submissions close to get certified?']


In [22]:
usage.input_tokens, usage.output_tokens

(207, 87)

In [23]:
from evaluation_utils import calc_price

In [24]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525,
 'output_cost': 0.00039150000000000003,
 'total_cost': 0.00054675}

In [25]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still join the course if I found it late?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course started, am I still eligible for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to get the certificate if I’m joining now?',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to start the course after it has already begun?',
  'document': '74eb249bbf'},
 {'question': 'Do I have to submit the project before submissions close to get certified?',
  'document': '74eb249bbf'}]

In [1]:
import pandas as pd

In [2]:
pd.DataFrame(records)

NameError: name 'records' is not defined